# 04 — E1: toy noise-smoothing bias

Runs `experiments/e1_toy_bias.py`: minimize `E_eps[R(softmax(z + eps))]` and show
that the **noise-free** `softmax(z*)` is systematically sharper than the target `t`,
with the gap growing in sigma (`docs/analysis/proof_sketch_smoothed_log_score.md`).

The full sweep is CPU-only but takes several minutes; the first cell below runs just
the fast baseline repl.</n

In [ ]:
from pathlib import Path
import os, sys

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("repo root:", REPO_ROOT)


In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location(
    "e1_toy_bias", REPO_ROOT / "experiments" / "e1_toy_bias.py"
)
e1 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(e1)
print("loaded E1 from", spec.origin)


## 1. Fast baseline repl (log-score only, t=[0.7, 0.2, 0.1])


In [ ]:
baseline = e1.run_baseline_replication()


## 2. Full experiment

Runs `baseline_replication` + `full_reward` + `k_sweep` and writes
`results/e1_toy_bias.json`. Set `RUN_FULL = True` when you are ready.


In [ ]:
RUN_FULL = False
if RUN_FULL:
    results = e1.main()
else:
    print("skipped — set RUN_FULL = True to run the full sweep")


## 3. Plot sharpening vs. sigma


In [ ]:
import json
import matplotlib.pyplot as plt

path = REPO_ROOT / "results" / "e1_toy_bias.json"
if path.exists():
    res = json.loads(path.read_text())

    b = res["baseline_replication"]
    sigmas = [r["sigma"] for r in b]
    plt.figure(figsize=(7, 4))
    plt.plot(sigmas, [r["sharpness_max_p"] for r in b], marker="o", label="noise-free max p")
    plt.axhline(b[0]["target"][0], ls="--", color="grey", label="target max")
    plt.xlabel("sigma")
    plt.ylabel("max probability")
    plt.title("baseline: over-confidence grows with sigma")
    plt.grid(True)
    plt.legend()
    plt.show()

    plt.figure(figsize=(7, 4))
    for k in sorted({r["k"] for r in res["k_sweep"]}):
        for reward in ("log_only", "full"):
            rows = [r for r in res["k_sweep"]
                    if r["k"] == k and r["reward"] == reward
                    and r["target_entropy_label"] == "peaked"]
            if rows:
                plt.plot([r["sigma"] for r in rows], [r["kl_p_from_t"] for r in rows],
                         marker="o", ls="-" if reward == "log_only" else "--",
                         label=f"K={k} {reward}")
    plt.xlabel("sigma")
    plt.ylabel("KL(target || p*)")
    plt.title("k_sweep (peaked targets)")
    plt.yscale("symlog", linthresh=1e-4)
    plt.grid(True, which="both")
    plt.legend(fontsize=8)
    plt.show()
else:
    print(f"{path} not found — run section 2 first")
